# 05 · Edge student (GDC-CNN) + distillation from the teacher (run on Kaggle GPU)

**Research question:** how much of the teacher's network knowledge (neighbours, wind, transport physics)
survives in a small student that only sees **one** station and runs on the ESP32?

| variant | the student learns from | loss |
|---|---|---|
| S0 | the true answer only | $L_{pred}$ |
| S1 | + the teacher's forecast | $\lambda L_{pred} + (1-\lambda) L_{out}$ (Shmuel et al.) |
| S2 | + the teacher's time features and attention | + $L_{feat} + L_{AFD}$ (Na et al.) |
| S3 | + the teacher's spatial state (GAT) | + $L_{spatial}$ |
| S4 | + the teacher's physics state (ADR) | + $L_{physics}$ |

S3 needs a teacher with a graph (T1-T3), S4 needs T3. Only the student goes to the ESP32;
the small helper layers used for distillation are thrown away.

## 0. Setup

In [ ]:
# imports
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers

In [ ]:
# where the data is (Kaggle or laptop)
ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    FEAT = next(Path("/kaggle/input").rglob("grid_u0.npz")).parent
    TEACHER_DIR = Path("/kaggle/input")                 # output of the 04 notebook is mounted here
    OUT = Path("/kaggle/working")
else:
    FEAT = Path("../data/4_features")
    TEACHER_DIR = Path("../data/5_models/teacher")
    OUT = Path("../data/5_models/student")
OUT.mkdir(parents=True, exist_ok=True)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# settings
SMOKE = not ON_KAGGLE                 # tiny quick run on the laptop
MODEL = "A"                           # "A" = SPS30 works, "B" = SPS30 broken (fallback model)
D = 16 if MODEL == "A" else 64        # student width (ESP32 size budget)
EPOCHS = 2 if SMOKE else 60
PATIENCE = 8
BATCH = 256
LAMBDA_OUT = 0.5                      # S1: half true answer, half teacher forecast
LAMBDA_PRED = 0.4                     # S2-S4: weight of the true answer (Na et al.)
WARMUP = 10                           # distillation weight grows from 0 to 1 over 10 epochs
TEACHERS = ["T0", "T1", "T2", "T3"]

## 1. Data
The student input of a window is cut from the same grids the teacher uses (one station, last 24 hours).

In [ ]:
# the 3 grids (only what the student needs: features and true PM)
grids = {u: {k: np.load(FEAT / f"grid_u{u}.npz")[k] for k in ("X", "pm")} for u in range(3)}
FEATURES = ["ln_pm25", "ln_pm10", "ln_pm1", "coarse_ratio", "t", "rh", "p", "wind_u", "wind_v", "rain_frac",
            "ln_lux", "mq7_n", "mq135_n", "voc_n", "nox_n", "hod_sin", "hod_cos", "dow_sin", "dow_cos",
            "doy_sin", "doy_cos", "aqi_now", "pm_mask"]
PM_FEATURES = ["ln_pm25", "ln_pm10", "ln_pm1", "coarse_ratio", "aqi_now", "pm_mask"]
USED = FEATURES if MODEL == "A" else [f for f in FEATURES if f not in PM_FEATURES]
print(MODEL, "uses", len(USED), "features")

In [ ]:
# the windows: which (station, unit, hour) are examples
windows_a = pd.read_parquet(FEAT / "windows_A.parquet")
windows_b = pd.read_parquet(FEAT / "windows_B.parquet")
kd_windows = pd.read_parquet(FEAT / "kd_windows.parquet")      # the windows the teacher made outputs for
EXTRAS = ["last_ln_pm25", "last_ln_pm10", "last_aqi", "hours_lost", "pm25_trend"]
len(windows_a), len(windows_b), len(kd_windows)

**Targets.** Model A: $y_h = \ln(1+PM[t+h]) - \ln(1+PM[t])$.
Model B (PM lost $k$ hours ago): relative to the last good hour, $y_h = \ln(1+PM[t+h]) - \ln(1+PM[t-k])$.

In [ ]:
# cut the 24-hour inputs and the 6 targets of a list of windows out of the grids
def get_xy(w):
    xs, ys = [], []
    for u, part in w.groupby("unit", sort=True):
        X, pm = grids[u]["X"], np.log1p(grids[u]["pm"])
        rows, st = part.row.to_numpy(), part.station.to_numpy()
        hours = rows[:, None] + np.arange(-23, 1)[None]                    # [n, 24]
        x = X[hours, st[:, None]][:, :, [FEATURES.index(f) for f in USED]]   # [n, 24, C]
        base = rows - (part.k.to_numpy() if MODEL == "B" else 0)            # hour the change is measured from
        y = np.concatenate([pm[rows + h, st, 0:1] - pm[base, st, 0:1] for h in (1, 2, 3)] +
                           [pm[rows + h, st, 1:2] - pm[base, st, 1:2] for h in (1, 2, 3)], axis=1)
        xs.append(x), ys.append(y)
    order = np.argsort(np.concatenate([part.index.to_numpy() for _, part in w.groupby("unit", sort=True)]))
    return np.concatenate(xs)[order].astype(np.float32), np.concatenate(ys)[order].astype(np.float32)

In [ ]:
# the 4 sets: distillation train / val (same windows for every variant), test, test at never-seen stations
pool = windows_a if MODEL == "A" else windows_b
if SMOKE:   # the smoke teacher (04) only made outputs for these 200 + 200 windows
    kd_windows = kd_windows.groupby("split").sample(200, random_state=0)
if MODEL == "A":
    train_w = kd_windows[kd_windows.split == "train"].reset_index(drop=True)
    val_w = kd_windows[kd_windows.split == "val"].reset_index(drop=True)
else:   # Model B: the B windows at the same (station, unit, hour) as the teacher's windows
    keys = ["station", "unit", "row"]
    joined = pool.merge(kd_windows[keys + ["window_id"]], on=keys)
    train_w, val_w = [joined[joined.split == s].reset_index(drop=True) for s in ("train", "val")]
test_w = pool[(pool.split == "test") & ~pool.holdout].sample(150_000, random_state=0).reset_index(drop=True)
unseen_w = pool[(pool.split == "test") & pool.holdout].sample(100_000, random_state=0).reset_index(drop=True)
if SMOKE:
    test_w, unseen_w = test_w.head(3000), unseen_w.head(3000)
len(train_w), len(val_w), len(test_w), len(unseen_w)

## 2. Preprocessing (the ESP32 does exactly this in C before calling the model)
1. **Instance norm:** each window, each feature: $\tilde x = (x - \mu_{24h}) / \sigma_{24h}$ — Delhi winter and summer look alike.
   (Not for rain, time-of-day codes and the PM mask.)
2. **Model B:** add the 5 "last good PM" numbers as constant channels.
3. **Patching:** 2 hours → one patch, so 24 h × C becomes 12 patches × 2C.
4. **Target scale:** the model predicts $y / s$ with $s = \max(\sigma_{24h}(\ln PM), 0.1)$; we multiply back by $s$.

In [ ]:
# instance norm + patches + target scale
NO_NORM = ["rain_frac", "hod_sin", "hod_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos", "pm_mask"]


def preprocess(x, extras=None):
    norm = np.array([f not in NO_NORM for f in USED])
    mean, std = x.mean(1, keepdims=True), np.sqrt(x.var(1, keepdims=True) + 1e-5)
    xn = np.where(norm, (x - mean) / std, x)
    if MODEL == "A":
        s25 = np.maximum(std[:, 0, USED.index("ln_pm25")], 0.1)
        s10 = np.maximum(std[:, 0, USED.index("ln_pm10")], 0.1)
        scale = np.repeat(np.stack([s25, s10], axis=1), 3, axis=1)
    else:
        xn = np.concatenate([xn, np.repeat(extras[:, None, :], 24, axis=1)], axis=2)
        scale = np.ones((len(x), 6))
    patches = xn.reshape(len(xn), 12, -1)
    return patches.astype(np.float32), scale.astype(np.float32)

In [ ]:
# build the arrays of every set
def make_set(w):
    x, y = get_xy(w)
    patches, scale = preprocess(x, w[EXTRAS].to_numpy(np.float32) if MODEL == "B" else None)
    return {"x": patches, "y": y, "scale": scale}


train, val, test, unseen = [make_set(w) for w in (train_w, val_w, test_w, unseen_w)]
train["x"].shape

## 3. The GDC-CNN student (Na et al. 2026)
```
12 patches -> Dense(D) + position -> [GDC block, dilation 1] -> [dilation 2] -> [dilation 4] -> flatten -> 6 outputs
GDC block:  LayerNorm -> causal dilated depthwise conv (kernel 3) -> Dense(2D) -> GLU -> Dense(D) -> + input
```
GLU (gated linear unit): $\text{GLU}(a, b) = a \odot \sigma(b)$ — the gate decides what passes.
Dilations 1, 2, 4 with kernel 3 let the last patch see all 12 patches. Causal = only looks at the past.

In [ ]:
# learned position code for the 12 patches
class PositionCode(layers.Layer):
    def build(self, shape):
        self.code = self.add_weight(shape=(shape[1], shape[2]), initializer="zeros", name="code")

    def call(self, x):
        return x + self.code

In [ ]:
# gated linear unit: first half x sigmoid(second half)
class GLU(layers.Layer):
    def call(self, x):
        half = x.shape[-1] // 2
        return x[..., :half] * keras.ops.sigmoid(x[..., half:])

In [ ]:
# one GDC block
def gdc_block(x, dilation):
    h = layers.LayerNormalization(epsilon=1e-5)(x)
    h = layers.ZeroPadding1D((2 * dilation, 0))(h)                        # causal: pad on the left only
    h = layers.DepthwiseConv1D(3, dilation_rate=dilation)(h)
    h = GLU()(layers.Dense(2 * D)(h))
    h = layers.Dense(D)(h)
    return layers.Add()([x, h])

In [ ]:
# the student; `features` gives the last block's output F_S [12, D] (needed for distillation)
def build_student(batch_size=None):
    inp = keras.Input((12, train["x"].shape[2]), batch_size=batch_size)
    x = PositionCode()(layers.Dense(D)(inp))
    for dilation in (1, 2, 4):
        x = gdc_block(x, dilation)
    y = layers.Dense(6)(layers.Flatten()(x))
    return keras.Model(inp, y), keras.Model(inp, [y, x])


student, _ = build_student()
student.summary()

## 4. S0: the student alone
Huber loss on $y/s$, Adam, early stopping on the validation loss.

In [ ]:
# train S0
keras.utils.set_random_seed(0)
s0, _ = build_student()
s0.compile(keras.optimizers.Adam(1e-3), loss=keras.losses.Huber())
s0.fit(train["x"], train["y"] / train["scale"], validation_data=(val["x"], val["y"] / val["scale"]),
       epochs=EPOCHS, batch_size=BATCH, verbose=2,
       callbacks=[keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)])

In [ ]:
# RMSE per output (ln change); persistence = "PM stays the same" (y = 0)
OUTPUTS = ["pm25_h1", "pm25_h2", "pm25_h3", "pm10_h1", "pm10_h2", "pm10_h3"]


def rmse(model, data):
    prediction = model.predict(data["x"], batch_size=4096, verbose=0) * data["scale"]
    return pd.Series(np.sqrt(((prediction - data["y"]) ** 2).mean(0)), index=OUTPUTS)


results = {"persistence": pd.Series(np.sqrt((test["y"] ** 2).mean(0)), index=OUTPUTS), "S0": rmse(s0, test)}
pd.DataFrame(results).round(4)

## 5. Teacher outputs for the student's windows
- States are z-scored (training statistics) so every loss term has a similar size.
- Model B: the teacher forecast is from hour $t$; we shift it to "relative to the last good hour":
  $y_T' = y_T + \ln(1+PM_t) - \ln(1+PM_{t-k})$.
- **Precondition (Shmuel et al.):** copy the teacher's forecast only for outputs where it beats S0 on validation.

In [ ]:
# load one teacher's outputs and line them up with our train / val windows
def load_teacher(name):
    path = next(TEACHER_DIR.rglob(f"teacher_{name}.npz"), None)
    if path is None:
        return None
    t = dict(np.load(path))
    position = pd.Series(np.arange(len(t["window_id"])), index=t["window_id"])
    arrays = {}
    for split, w, data in (("train", train_w, train), ("val", val_w, val)):
        i = position.reindex(w.window_id).to_numpy()
        assert not np.isnan(i).any(), "teacher has no output for some windows"
        i = i.astype(int)
        y_t = t["y_teacher"][i].astype(np.float32)
        if MODEL == "B":
            u_rows = [(u, r, s, k) for u, r, s, k in zip(w.unit, w.row, w.station, w.k)]
            shift = np.array([[np.log1p(grids[u]["pm"][r, s, p]) - np.log1p(grids[u]["pm"][r - k, s, p]) for p in (0, 1)]
                              for u, r, s, k in u_rows], np.float32)
            y_t = y_t + np.repeat(shift, 3, axis=1)
        arrays[split] = {"y_t": y_t, "f_t": t["feat12"][i].astype(np.float32), "afd": t["afd12"][i].astype(np.float32),
                         "h_spatial": t["h_spatial"][i].astype(np.float32), "h_adr": t["h_adr"][i].astype(np.float32)}
    for key in ("f_t", "h_spatial", "h_adr"):                         # z-score with training statistics
        axes = (0, 1) if key == "f_t" else 0
        mu, sd = arrays["train"][key].mean(axes), arrays["train"][key].std(axes) + 1e-6
        for split in arrays:
            arrays[split][key] = (arrays[split][key] - mu) / sd
    teacher_val_rmse = np.sqrt(((arrays["val"]["y_t"] - val["y"]) ** 2).mean(0))
    gate = (teacher_val_rmse < rmse(s0, val).to_numpy()).astype(np.float32)
    return arrays, gate

## 6. Distillation losses
All in the real ln-change space (prediction × $s$). Huber $\ell(a, b)$ everywhere.

| term | formula | meaning |
|---|---|---|
| $L_{pred}$ | $\ell(\hat y, y)$ | match the truth |
| $L_{out}$ | $\ell(\hat y, y_T)$ (only gated outputs) | match the teacher's forecast |
| $L_{feat}$ | $\ell(F_S,\ \psi(F_T))$ | student time features ≈ teacher time features (12 patches) |
| $L_{AFD}$ | $\sum_p a_p\,\ell(F_{S,p}, \psi(F_{T,p}))$ | same, weighted by the teacher's attention $a_p$ |
| $L_{spatial}$ | $\ell(P_s(\bar F_S),\ h_{spatial})$ | student guesses the teacher's neighbour state |
| $L_{physics}$ | $\ell(P_p(\bar F_S),\ h_{adr})$ | student guesses the teacher's physics state |

Total for S2-S4 (Na et al. + our extra terms, $n$ = number of active distillation terms, $r$ = warm-up 0→1):
$$L = \lambda_p L_{pred} + \frac{1-\lambda_p}{n}\, r(e) \sum L_{kd}$$
$\psi$, $P_s$, $P_p$ are small Dense layers used only in training.

In [ ]:
# Huber error
def huber(a, b):
    d = tf.abs(a - b)
    return tf.where(d <= 1.0, 0.5 * d * d, d - 0.5)

In [ ]:
# all loss terms of one batch for one variant
def kd_losses(variant, nets, b, gate, warmup, training):
    y_norm, f_s = nets["features"](b["x"], training=training)
    y_hat = y_norm * b["scale"]
    L = {"pred": tf.reduce_mean(huber(y_hat, b["y"]))}
    if variant == "S1":
        L["out"] = tf.reduce_mean(huber(y_hat, b["y_t"]) * gate)
        return LAMBDA_OUT * L["pred"] + (1 - LAMBDA_OUT) * L["out"]
    L["out"] = tf.reduce_mean(huber(y_hat, b["y_t"]) * gate)
    error = huber(f_s, nets["psi"](b["f_t"]))                            # [batch, 12, D]
    L["feat"] = tf.reduce_mean(error)
    L["afd"] = tf.reduce_mean(tf.reduce_sum(b["afd"][:, :, None] * error, axis=1))
    pooled = tf.reduce_mean(f_s, axis=1)
    if variant in ("S3", "S4"):
        L["spatial"] = tf.reduce_mean(huber(nets["p_spatial"](pooled), b["h_spatial"]))
    if variant == "S4":
        L["physics"] = tf.reduce_mean(huber(nets["p_physics"](pooled), b["h_adr"]))
    kd_terms = [v for k, v in L.items() if k != "pred"]
    return LAMBDA_PRED * L["pred"] + (1 - LAMBDA_PRED) / len(kd_terms) * warmup * tf.add_n(kd_terms)

In [ ]:
# train one distilled student (S1-S4); early stopping on validation RMSE against the TRUE target
def train_kd(variant, t_train, t_val, gate):
    keras.utils.set_random_seed(0)
    student, features = build_student()
    d_teacher = t_train["f_t"].shape[-1]
    nets = {"student": student, "features": features, "psi": layers.Dense(D),
            "p_spatial": layers.Dense(d_teacher), "p_physics": layers.Dense(d_teacher)}
    nets["psi"].build((None, d_teacher)), nets["p_spatial"].build((None, D)), nets["p_physics"].build((None, D))
    weights = [w for k, n in nets.items() if k != "features" for w in n.trainable_variables]
    opt = keras.optimizers.Adam(1e-3)
    data = {**train, **t_train}
    gate = tf.constant(gate)

    @tf.function
    def step(b, warmup):
        with tf.GradientTape() as tape:
            loss = kd_losses(variant, nets, b, gate, warmup, True)
        grads = tape.gradient(loss, weights)
        opt.apply_gradients([(g, w) for g, w in zip(grads, weights) if g is not None])

    best, best_weights, wait = np.inf, None, 0
    rng = np.random.default_rng(0)
    for epoch in range(EPOCHS):
        order = rng.permutation(len(data["y"]))
        warmup = tf.constant(min(1.0, epoch / WARMUP), tf.float32)
        for k in range(0, len(order), BATCH):
            step({key: tf.constant(v[order[k:k + BATCH]]) for key, v in data.items()}, warmup)
        score = rmse(student, val).mean()
        print(f"{variant} epoch {epoch}: val RMSE {score:.4f}")
        if score < best:
            best, best_weights, wait = score, student.get_weights(), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    student.set_weights(best_weights)
    return student

## 7. Run all variants

In [ ]:
# which teacher can be used for which variant
ALLOWED = {"S1": TEACHERS, "S2": TEACHERS, "S3": ["T1", "T2", "T3"], "S4": ["T3"]}
if MODEL == "B":
    ALLOWED = {"S1": TEACHERS}            # plan: output distillation first for Model B

In [ ]:
# train S1-S4 with every teacher
students = {"S0": s0}
unseen_results = {"persistence": pd.Series(np.sqrt((unseen["y"] ** 2).mean(0)), index=OUTPUTS), "S0": rmse(s0, unseen)}
for teacher in TEACHERS:
    loaded = load_teacher(teacher)
    if loaded is None:
        print("no outputs for", teacher)
        continue
    t_arrays, gate = loaded
    print(teacher, "beats S0 on", OUTPUTS if gate.all() else [o for o, g in zip(OUTPUTS, gate) if g])
    for variant, allowed in ALLOWED.items():
        if teacher in allowed:
            name = f"{variant}-{teacher}"
            students[name] = train_kd(variant, t_arrays["train"], t_arrays["val"], gate)
            results[name] = rmse(students[name], test)
            unseen_results[name] = rmse(students[name], unseen)

In [ ]:
# results: test RMSE (ln change) and change vs S0 in %
table = pd.DataFrame(results).T
table["mean"] = table[OUTPUTS].mean(axis=1)
table["vs S0 %"] = 100 * (table["mean"] / table.loc["S0", "mean"] - 1)
table.to_csv(OUT / f"student_{MODEL}_results.csv")
table.round(4)

In [ ]:
# the same at the 5 stations never used for training
unseen_table = pd.DataFrame(unseen_results).T
unseen_table["mean"] = unseen_table[OUTPUTS].mean(axis=1)
unseen_table.round(4)

In [ ]:
# plot: PM2.5 error by forecast hour for every student
plt.figure(figsize=(9, 4))
for name, row in table.iterrows():
    plt.plot([1, 2, 3], row[OUTPUTS[:3]], marker="o", label=name, ls="--" if name == "persistence" else "-")
plt.xlabel("hours ahead")
plt.ylabel("RMSE (ln change)")
plt.title(f"Model {MODEL}: student test error, PM2.5")
plt.legend(fontsize=7, ncol=3)
plt.show()

## 8. Export the best student for the ESP32 (TFLite, FP32)
Rebuilt with batch size 1 (fixed shapes for TFLite Micro), converted, and checked: it must give the
same numbers as the Keras model.

In [ ]:
# pick the best student on validation and convert it
best_name = min(students, key=lambda n: rmse(students[n], val).mean())
export_model, _ = build_student(batch_size=1)
export_model.set_weights(students[best_name].get_weights())
tflite_bytes = tf.lite.TFLiteConverter.from_keras_model(export_model).convert()
(OUT / f"student_{MODEL}_{best_name}.tflite").write_bytes(tflite_bytes)
print(best_name, f"{len(tflite_bytes) / 1024:.1f} KB")

In [ ]:
# check: TFLite gives the same output as Keras
interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
interpreter.allocate_tensors()
inp, out = interpreter.get_input_details()[0]["index"], interpreter.get_output_details()[0]["index"]
worst = 0.0
for x in val["x"][:200]:
    interpreter.set_tensor(inp, x[None])
    interpreter.invoke()
    worst = max(worst, float(np.abs(interpreter.get_tensor(out) - export_model(x[None]).numpy()).max()))
print(f"largest difference TFLite vs Keras: {worst:.1e}")
assert worst < 1e-4